[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/superconductivity/superconductivity.ipynb)

# A superconducting condensate rings

Change the attraction between paired electrons and the superconducting gap oscillates. Each pair of opposite-momentum orbitals has a pseudospin: occupation lies along z, pairing coherence in the x–y plane. Their collective coherence supplies the field that turns them, so the gap evolves with the spins. These directions describe pairing states, not spatial electron spins.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from functools import partial

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.quantum.superconductivity import render
from examples.quantum.superconductivity.core import energy_levels, evolution

In [ ]:
ga = VGA3D
mv = NumpyContext(ga).multivector
Spin = ga.gatype.vector()
Map = ga.gatype((Spin, Spin))                              # Spin <- Spin

The zero-temperature ground state points each spin against its effective field, with length one half. But that field includes the gap produced by those same spins. Repeated alignment solves this self-consistency condition. We use equally weighted energy levels, a constant density of states, and one energy unit $E_0$ with $\hbar=1$.

In [ ]:
levels = 240
cutoff = 5.0
coupling = 4.4
alignment_iterations = 160
seed = mv.x

dispersion = energy_levels(levels, cutoff)                 # [levels] Spin
# The open Spin reads any input; subtracting its z part leaves pairing coherence.
transverse = Spin - mv.z * (mv.z | Spin)                   # [] Spin <- Spin
pairing = coupling * transverse                          # [] Spin <- Spin


def ground_state(pairing: Map, seed: Spin) -> Spin:
    gap = seed
    for _ in range(alignment_iterations):
        spins = -(dispersion - gap).normalized() / 2
        gap = pairing(spins.mean(axis=-1, keepdims=True))  # [1] Spin
    return spins


initial = ground_state(pairing, seed)                     # [levels] Spin
render.draw_equilibrium(dispersion, initial);

Reduce the attraction suddenly. The spins cannot jump, but the gap they produce immediately changes with the coupling. The resulting field turns them through a bivector commutator, and their changing mean feeds back into that field at every step. The arrows show the pairing-plane projection at sampled energies; the gap uses all levels.

In [ ]:
quench_ratio = 0.88
post_pairing = quench_ratio * pairing
dt = 0.01
steps = 2400
stride = 10
midpoint_iterations = 10
duration_ms = 50
times = np.arange(steps // stride + 1) * stride * dt


def precession(spins: Spin, pairing: Map) -> Spin:
    gap = pairing(spins.mean(axis=-1, keepdims=True))      # [..., 1] Spin
    field = dispersion - gap                             # [..., levels] Spin
    return (-2 * mv.xyz * field).commutator(spins)


# Implicit midpoint evaluates the field and the spins halfway through each step.
# Its converged update preserves spin lengths, total occupation and the BCS energy.
history = stack(evolution(initial, partial(precession, pairing=post_pairing),
                          dt, steps, stride, midpoint_iterations))  # [times, levels] Spin
gaps = post_pairing(history.mean(axis=-1, keepdims=True))  # [times, 1] Spin
render.inline(render.animate_quench(dispersion, history, gaps, times), duration_ms)

The gap rings while the spins spread in phase. Its decreasing oscillation amplitude is dephasing within an isolated system; no energy is removed. The dashed line below is the ground-state gap at the final attraction, which differs from the value approached after the quench. This amplitude oscillation is the superconducting Higgs response.

In [ ]:
final_equilibrium = ground_state(post_pairing, seed)
final_gap = post_pairing(final_equilibrium.mean(axis=-1, keepdims=True))
render.draw_gap(times, gaps, final_gap);

A small disturbance changes both the spin being turned and the field doing the turning. Leave the disturbance open in each place: one unary map acts on the local spin change, the other on the mean change across all energies. Keeping both predicts the collective gap response; holding the pairing field fixed misses that feedback.

In [ ]:
small_quench_ratio = 0.99
weak_pairing = small_quench_ratio * pairing
feedback_strengths = np.array([1.0, 0.0])
response_names = ("collective response", "fixed pairing field")

equilibrium = ground_state(weak_pairing, seed)             # [levels] Spin
equilibrium_gap = weak_pairing(equilibrium.mean(axis=-1, keepdims=True))
field = dispersion - equilibrium_gap

# Vary the spin at fixed field, then vary the field at fixed spin.
local = (-2 * mv.xyz * field).commutator(Spin)             # [levels] Spin <- Spin
feedback = (2 * mv.xyz * weak_pairing).commutator(equilibrium)
feedback = feedback * feedback_strengths[:, None]         # [cases, levels] Spin <- Spin


def response(delta: Spin) -> Spin:
    return local(delta) + feedback(delta.mean(axis=-1, keepdims=True))


# Both approximations start from the same small departure from the final ground state.
disturbance = (initial - equilibrium).broadcast_to((len(feedback_strengths), levels))
linear_history = stack(evolution(disturbance, response, dt, steps, stride, midpoint_iterations))
weak_history = stack(evolution(initial, partial(precession, pairing=weak_pairing),
                               dt, steps, stride, midpoint_iterations))
exact_gap_change = weak_pairing(weak_history.mean(axis=-1, keepdims=True)) - equilibrium_gap
predicted_gap_change = weak_pairing(linear_history.mean(axis=-1, keepdims=True))
render.draw_response(times, exact_gap_change, predicted_gap_change, response_names);

This is the collisionless, spatially uniform, zero-temperature reduced BCS mean-field model with s-wave pairing. It evolves the order parameter self-consistently, but includes neither collisions nor electromagnetic screening, vortices or a microscopic account of the attractive interaction. A coupling quench is a simple way to excite the amplitude response; modelling an optical pump would require its coupling to the electron dispersion. [Chou, Liao and Foster](https://arxiv.org/abs/1611.07089) connect pseudospin dynamics and gap oscillations to ultrafast experiments; [Tsuji and Aoki](https://arxiv.org/abs/1404.2711) study optical excitation of the Higgs mode.